# DAF-07: The First Trained Model, Ridge inside a Pipeline

Asha has a free forecasting rule: at 18:00 IST, assume tomorrow will look like
today. In DAF-06 we measured it. Her friend Ravi asks: *"Why not let a model
learn the rule from the data?"*

This notebook trains one model, `Ridge`, and scores it against Asha's rule on
the **same test days**. The question is:

**Can a trained model beat a free guess?**

Two rules keep the answer honest:

- every input must be known at 18:00 today, nothing from tomorrow
- the model and its scaler learn from the training days only


## 1. Load the daily table and reuse the DAF-06 cut date

We use the same table and the same cut date as DAF-06. If we changed the cut
date, the scores would not be comparable with the baselines.

```text
before 2026-03-01  ->  train (the model learns from these)
from   2026-03-01  ->  test  (the model never sees these while learning)


In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
daily_all = pd.read_csv(
    project_root / "data/interim/daily_17.csv",
    parse_dates=["date"],
    index_col="date",
)

cut_date = pd.Timestamp("2026-03-01", tz=daily_all.index.tz)
poor_threshold = 91   # same Poor-or-worse line as DAF-06

print("Rows:", len(daily_all))
print("Date range:", daily_all.index.min().date(), "to", daily_all.index.max().date())
print("Cut date:", cut_date.date())
print("Days before the cut:", (daily_all.index < cut_date).sum())
print("Days from the cut:", (daily_all.index >= cut_date).sum())
daily_all.head()

Rows: 571
Date range: 2025-02-19 to 2026-09-12
Cut date: 2026-03-01
Days before the cut: 375
Days from the cut: 196


,pm25_mean,hours,pm25_until_17,valid,target
date,,,,,
2025-02-19 00:00:00+05:30,78.782609,23,79.117647,True,49.416667
2025-02-20 00:00:00+05:30,49.416667,24,49.611111,True,90.375000
2025-02-21 00:00:00+05:30,90.375000,24,88.222222,True,69.125000
2025-02-22 00:00:00+05:30,69.125000,24,69.777778,True,74.000000
2025-02-23 00:00:00+05:30,74.000000,24,70.277778,True,101.166667


## 2. Build the features, using only what is known at 18:00

### The one question this step answers

It is 18:00 on 10 March. Asha wants to forecast 11 March. **Which numbers does
she really know right now?**

| Feature | What it is | Known at 18:00? |
|---|---|---|
| `pm25_until_17` | today's average, hours 00-17 | Yes, today is half done |
| `lag_1` | yesterday's full-day mean (9 March) | Yes, yesterday is finished |
| `mean_3` | mean of the last 3 full days (7, 8, 9 March) | Yes |
| `mean_7` | mean of the last 7 full days (3 to 9 March) | Yes |
| `month` | 3 | Yes, it is the calendar |

What she does **not** know is today's full-day mean (`pm25_mean` for 10 March),
because the day is not over. It also must not appear in any feature.

### The trap: a rolling mean that includes today

`rolling(3).mean()` on the row for 10 March averages **8, 9 and 10 March**. That
includes today's full day, which is not finished at 18:00. It would be like
predicting an exam result after seeing half the answers.

The fix is to slide the numbers down by one row **first**, and then roll:

```text
date      pm25_mean   shift(1)     shift(1).rolling(3).mean()
Mar 7        100         NaN              NaN
Mar 8        120         100              NaN
Mar 9         80         120              NaN
Mar 10        90          80      (100+120+80)/3 = 100   <- uses Mar 7, 8, 9 only
```

Now the row for 10 March only looks at days that are **already finished**.

### What we do

1. Build the five features and the `target` on one table.
2. Check one row by hand, so we know the shift is right.
3. Drop the rows where any value is missing, and count how many that costs.


In [5]:
# Yesterday's full-day mean, on every row. Row "Mar 10" now holds Mar 9's value.
yesterday = daily_all["pm25_mean"].shift(1)

features = pd.DataFrame(index=daily_all.index)
features["pm25_until_17"] = daily_all["pm25_until_17"]
features["lag_1"] = yesterday
features["mean_3"] = yesterday.rolling(3).mean()
features["mean_7"] = yesterday.rolling(7).mean() 
features["month"] = daily_all.index.month
features["target"] = daily_all["target"]

feature_names = ["pm25_until_17", "lag_1", "mean_3", "mean_7", "month"]

# Hand check: mean_3 for Mar 10 must be the mean of Mar 7, 8, 9 (NOT Mar 10)
check_day = pd.Timestamp("2026-03-10", tz=daily_all.index.tz)
by_hand = daily_all["pm25_mean"].loc["2026-03-07":"2026-03-09"].mean()
print("mean_3 in the table :", round(features.loc[check_day, "mean_3"], 3))
print("mean of Mar 7, 8, 9 :", round(by_hand, 3))
assert abs(features.loc[check_day, "mean_3"] - by_hand) < 1e-9, "mean_3 is not ending yesterday"

features.loc["2026-03-08":"2026-03-12"]


mean_3 in the table : 129.231
mean of Mar 7, 8, 9 : 129.231


,pm25_until_17,lag_1,mean_3,mean_7,month,target
date,,,,,,
2026-03-08 00:00:00+05:30,111.166667,148.400000,101.563889,94.873499,3,117.250000
2026-03-09 00:00:00+05:30,116.500000,122.041667,119.313889,99.829762,3,NaN
2026-03-10 00:00:00+05:30,76.800000,117.250000,129.230556,102.301190,3,117.041667
2026-03-11 00:00:00+05:30,117.111111,111.272727,116.854798,104.560390,3,104.217391
2026-03-12 00:00:00+05:30,98.235294,117.041667,115.188131,110.328247,3,90.500000


In [9]:
# Three reasons a row cannot be used
no_target = features["target"].isna()                       # no tomorrow to score against
no_feature = features[feature_names].isna().any(axis=1)     # some input is missing

print("Rows in the table:", len(features))
print("Rows with no target:", no_target.sum())
print("Rows with a missing feature (and a target):", (no_feature & ~no_target).sum())

model_rows = features.dropna()
print("\nRows we can use:", len(model_rows))
print("Rows lost in total:", len(features) - len(model_rows))

# Which rows did we lose to the missing features? Show the dates.
lost_to_features = features[no_feature & ~no_target]
print("\nLost only because a feature is missing:")
print(lost_to_features[feature_names].round(1).to_string())

Rows in the table: 571
Rows with no target: 84
Rows with a missing feature (and a target): 59

Rows we can use: 428
Rows lost in total: 143

Lost only because a feature is missing:
                           pm25_until_17  lag_1  mean_3  mean_7  month
date                                                                  
2025-02-19 00:00:00+05:30           79.1    NaN     NaN     NaN      2
2025-02-20 00:00:00+05:30           49.6   78.8     NaN     NaN      2
2025-02-21 00:00:00+05:30           88.2   49.4     NaN     NaN      2
2025-02-22 00:00:00+05:30           69.8   90.4    72.9     NaN      2
2025-02-23 00:00:00+05:30           70.3   69.1    69.6     NaN      2
2025-02-24 00:00:00+05:30          102.7   74.0    77.8     NaN      2
2025-02-25 00:00:00+05:30           98.7  101.2    81.4     NaN      2
2025-03-03 00:00:00+05:30           49.9    NaN     NaN     NaN      3
2025-03-04 00:00:00+05:30           55.7   52.4     NaN     NaN      3
2025-03-05 00:00:00+05:30           29

### 2.3 Split by time and check the overlap with the baselines

We use the same cut date as DAF-06:

```text
model_rows  ->  before 2026-03-01  ->  train
            └-> from   2026-03-01  ->  test
```

Dropping incomplete rows may have removed some test days. The baselines were
scored on 161 days. A fair comparison needs **the same days for everyone**, so
we check how many test days the model has, and how many of them are also in the
baseline set.


### Map of the objects so far

Think of each name as a **labelled box** in the notebook's memory. Nothing new
is downloaded. Each box is built from an earlier box.

```text
FILE on disk
data/interim/daily_17.csv
        │  pd.read_csv
        ▼
┌───────────────────────────────────────────────────────────────┐
│ daily_all      table, 571 rows x 5 columns (ONE ROW PER DAY)  │
│   pm25_mean · hours · pm25_until_17 · valid · target          │
└───────────────────────────────────────────────────────────────┘
        │                                   │
        │ .shift(1)                         │ copy columns
        ▼                                   ▼
┌───────────────────┐        ┌───────────────────────────────────┐
│ yesterday         │ ─────► │ features   table, 571 rows x 6    │
│ ONE column, 571   │ lag_1  │  pm25_until_17 · lag_1 · mean_3   │
│ (yesterday's mean)│ mean_3 │  mean_7 · month · target          │
└───────────────────┘ mean_7 └───────────────────────────────────┘
                                        │ .dropna()
                                        ▼
                             ┌───────────────────────┐
                             │ model_rows  428 rows  │  <- only complete rows
                             └───────────────────────┘
                                   │ by date        │
                                   ▼                ▼
                             train (283 rows)   test (145 rows)
```

| Name | What it is | Size |
|---|---|---|
| `project_root` | folder path of the project | one path |
| `daily_all` | the whole daily table from DAF-05 | 571 x 5 |
| `cut_date` | the date that separates train from test | one date |
| `poor_threshold` | the Poor-or-worse line, 91 | one number |
| `yesterday` | yesterday's full-day mean, aligned to today's row | 571 values |
| `feature_names` | just the 5 column names, as a list | 5 names |
| `features` | the 5 inputs plus the answer (`target`), all rows | 571 x 6 |
| `no_target`, `no_feature` | True/False per row: "is something missing?" | 571 each |
| `model_rows` | `features` with incomplete rows removed | 428 x 6 |
| `train`, `test` | `model_rows` cut at `cut_date` | 283 and 145 |

**Java comparison:** `daily_all` is the raw table. `features` is a
`SELECT ... FROM daily_all` with new computed columns. `model_rows` is the same
query with `WHERE no column IS NULL`. `train` and `test` are two `WHERE date <`
and `WHERE date >=` views of it.

### What is `baseline_days`?

In DAF-06 the two baselines were scored on **161 test days**. A day was only
scored if all three things existed:

```text
1. a real answer for tomorrow       (target)
2. persistence's guess              (pm25_until_17)
3. yesterday-rule's guess           (yesterday's mean)
```

`baseline_days` is just **the list of those 161 dates**. We need it because the
model has fewer test days (145). Comparing MAE on different days is unfair, like
comparing two students who were given different exam papers. Later we score all
three methods on the days they have **in common**.


### What we learned: the model and the baselines have different test days

The DAF-06 baselines were scored on **161** test days. Our model can only be
tested on **145** of them, because `mean_7` needs 7 finished days in a row and
the sensor data has gaps.

`baseline_days` is only a list of dates: the days DAF-06 scored. A day needed
three things: a real answer for tomorrow, persistence's guess, and the
yesterday-rule's guess.

```text
Example with 6 test days

Day   baselines can guess?   model has all 5 features?
Mon         yes                    yes
Tue         yes                    yes
Wed         yes                    NO  (mean_7 empty)
Thu         yes                    yes
Fri         yes                    NO  (mean_7 empty)
Sat         yes                    yes

baselines scored on: Mon Tue Wed Thu Fri Sat   (6 days)
model can predict  : Mon Tue     Thu     Sat   (4 days)
```

### Why the days must be the same

An average error depends on **how hard the days are**. Suppose Wed and Fri are
hard days where the baseline is wrong by 20, and on the others it is wrong by 5:

```text
Baseline on 6 days: (5 + 5 + 20 + 5 + 20 + 5) / 6 = 10
Baseline on the model's 4 days: (5 + 5 + 5 + 5) / 4 =  5
```

The same guesser scores 10 or 5 depending on the days. Two students who sat
different exam papers cannot be compared by their averages.

### What we found in our data

| Check | Result |
|---|---|
| Days the baselines were scored on | 161 |
| Days the model can be tested on | 145 |
| Model days that are also baseline days | 145 (all of them) |
| Baseline days the model lost | 16 |
| Poor-or-worse days in the model's test rows | 17 (none lost) |

The 145 model days are a **subset** of the 161 baseline days, so every model
day has a baseline guess. In Step 4 we re-score persistence and yesterday on
these same 145 days. The numbers will differ slightly from DAF-06's 15.73 and
16.40, and that is expected. All 17 Poor days are still in the test set, so
recall stays comparable.


In [11]:
# ── Step 2.3  Split the usable rows by time ─────────────────────────────────
train = model_rows[model_rows.index < cut_date]     # older days: the model may learn from these
test = model_rows[model_rows.index >= cut_date]     # newer days: the model must not see these

print("Train rows:", len(train), "| from", train.index.min().date(), "to", train.index.max().date())
print("Test rows :", len(test), "| from", test.index.min().date(), "to", test.index.max().date())

# ── Which days did the DAF-06 baselines score? ──────────────────────────────
# A baseline day needs three things, so we check each one separately.

after_cut = daily_all[daily_all.index >= cut_date]        # every day from the cut date onward

has_target = after_cut["target"].notna()                  # 1. we know the real answer for tomorrow
has_persistence = after_cut["pm25_until_17"].notna()      # 2. persistence has a guess (today's 00-17 mean)
has_yesterday = yesterday.reindex(after_cut.index).notna()  # 3. the yesterday rule has a guess

# Keep the dates where all three are True. This is the list of days DAF-06 scored.
baseline_days = after_cut.index[has_target & has_persistence & has_yesterday]

# ── Compare the two lists of dates ──────────────────────────────────────────
print("\nDays the baselines were scored on :", len(baseline_days))
print("Days the model can be tested on   :", len(test))
print("Days in BOTH lists                :", test.index.isin(baseline_days).sum())
print("Baseline days the model lost      :", (~baseline_days.isin(test.index)).sum())

# Poor-or-worse days the model will be tested on
print("\nPoor-or-worse days in model test rows:", (test["target"] >= poor_threshold).sum())


Train rows: 283 | from 2025-02-26 to 2026-02-28
Test rows : 145 | from 2026-03-01 to 2026-09-10

Days the baselines were scored on : 161
Days the model can be tested on   : 145
Days in BOTH lists                : 145
Baseline days the model lost      : 16

Poor-or-worse days in model test rows: 17


## 3. Train Ridge inside a Pipeline

### The one question this step answers

We have 283 training days and 5 inputs for each. **How does the model learn
from them, without ever peeking at the test days?**

### Two small ideas

**1. Scaling.** The five features are on very different sizes: `month` is 1 to
12, while `pm25_until_17` can be 200. `StandardScaler` puts every feature on the
same footing by asking "how far from normal is this value?"

```text
new value = (value - mean) / spread

pm25_until_17 = 100, train mean = 80, train spread = 20
new value = (100 - 80) / 20 = 1.0     "one spread above normal"
```

Ridge adds a penalty on the size of each coefficient. Without scaling, that
penalty would be unfair to features with small numbers.

**2. Ridge.** It is a straight-line model, like `LinearRegression`, with one
extra rule: *keep the coefficients small*. When `lag_1`, `mean_3` and `mean_7`
say almost the same thing, plain regression can give one a huge positive and
another a huge negative value. Ridge keeps them calm. We leave the strength
`alpha` at its default (1.0). Tuning it is DAF-17.

### Why a Pipeline

```text
Pipeline([ ("scale", StandardScaler()), ("model", Ridge()) ])

pipe.fit(X_train, y_train):
    scaler learns mean and spread   <- from TRAIN rows only
    train rows are scaled
    Ridge learns from the scaled train rows

pipe.predict(X_test):
    scaler REUSES the train mean and spread   <- it does not learn again
    test rows are scaled with those numbers
    Ridge predicts
```

If we scaled the whole table first, the mean and spread would include the test
days. The model would then have seen a little of the exam before sitting it.
This is called **data leakage**. The Pipeline makes leakage impossible, because
`fit` only ever touches the train rows.

**Java comparison:** a Pipeline is like a service that owns its own
configuration. You build it once, and it applies the same steps in the same
order at training time and at prediction time.


### What is a Pipeline?

A Pipeline is **a list of steps that always run in the same order, as one
object**. We have two steps: scale, then predict.

```text
                       pipe = Pipeline([ scale , model ])

   raw features            step 1: "scale"             step 2: "model"
  ┌────────────┐        ┌──────────────────┐        ┌──────────────────┐
  │ 100, 110,  │  ───►  │  StandardScaler  │  ───►  │      Ridge       │  ───►  tomorrow's
  │ 105, 98, 3 │        │ puts every input │        │ multiplies each  │        PM2.5 guess
  └────────────┘        │ on the same scale│        │ input by a weight│
   (5 numbers,          └──────────────────┘        └──────────────────┘
   all different sizes)   output: about -1 to +1      output: 1 number
```

### The two moments in a pipeline's life

A pipeline does different work when **learning** and when **predicting**. This
is the most important thing to understand about it.

```text
LEARNING:  pipe.fit(X_train, y_train)          ← only the TRAIN days go in

   X_train ──► scaler LEARNS its mean and spread ──► scales X_train ──► Ridge LEARNS its weights
                    │                                                         │
                    └── stored inside the pipeline ──────────────────────────┘


PREDICTING:  pipe.predict(X_test)              ← the TEST days go in

   X_test ──► scaler REUSES the stored mean and spread ──► Ridge REUSES its weights ──► guess
              (it does NOT learn again from the test days)
```

| Moment | Scaler does | Ridge does | Data it may look at |
|---|---|---|---|
| `fit` (learning) | learns the mean and spread | learns the weights | **train days only** |
| `predict` (using) | reuses the stored mean and spread | reuses the stored weights | test days go through, nothing is learned from them |

### A worked example of the scaler, with our real numbers

The scaler learned from the train days that a normal `pm25_until_17` is about
**109** with a spread of about **87**. For one test day with `pm25_until_17 = 200`:

```text
scaled value = (value - train mean) / train spread
             = (200 - 108.9) / 86.75
             = 1.05         "a bit more than one spread above the train normal"
```

Ridge then works on that `1.05`, and never on the raw 200. Every feature is
converted the same way, so `month` (values 1 to 12) and `pm25_until_17` (values
up to 300) become comparable.

### Why use a Pipeline at all?

**Without a pipeline** you would write the steps by hand, and one small mistake
leaks the test days:

```text
WRONG (leaks)                                   RIGHT (pipeline)
scaler.fit(ALL days)   ← test days included     pipe.fit(X_train, y_train)
X_scaled = scaler.transform(ALL days)           pipe.predict(X_test)
split into train / test                         ← the scaler never sees test at fit time
model.fit(train) ...
```

A Pipeline makes the right thing the only thing you can do. It is like a
service class that owns its configuration: build it once, and it applies the same
steps in the same order in training and in production. Later, when we save the
model for the API (DAF-19 onward), we save **one object**, so the scaler cannot
get separated from the model.


### Why did we build `mean_check`?

A Pipeline promises: *"the scaler learns from the train days only."* Promises in
code should be **tested**, in the same way you write a unit test for a method
that claims to be thread-safe. `mean_check` is that test.

### What it shows

Each column is the average of each feature, from a different source:

| Column | Where the number comes from | What we expect |
|---|---|---|
| `scaler learned` | `scaler.mean_`, the number the pipeline stored | the value to test |
| `train mean` | `X_train.mean()`, computed by hand from the train days | **must equal** `scaler learned` |
| `test mean` | `X_test.mean()`, computed by hand from the test days | **must differ** from `scaler learned` |

```text
Our real numbers:

feature          scaler learned    train mean    test mean
pm25_until_17         108.91         108.91         55.53
lag_1                 114.28         114.28         57.47
mean_3                113.36         113.36         57.94
mean_7                111.89         111.89         59.13
month                   7.18           7.18          5.28
                        └── identical ──┘        └── different
```

### How to read the result

```text
scaler learned == train mean   ->  the scaler looked at the train days.  ✅ no leakage
scaler learned == test  mean   ->  the scaler looked at the test days.   ❌ leakage
scaler learned == all-days mean ->  it looked at everything.             ❌ leakage
```

The test would only be meaningful if the train and test means are **different**.
If they were nearly equal, a match would prove nothing, because we could not tell
which set the scaler used. Ours are clearly different (109 against 56), so the
check is real.

### The `assert` line

```python
assert (abs(scaler.mean_ - X_train.mean().values) < 1e-9).all(), "scaler did not learn from train"
```

`assert` is Python's guard clause. If the condition is false, the notebook
**stops with an error** instead of carrying on with a broken result. It is like
`Objects.requireNonNull`. It means that if someone later changes the code so the
scaler sees the test days, this cell fails loudly instead of silently giving a
flattering score.

### A finding hiding in this table

The average PM2.5 in the train days is about **109**, and in the test days about
**56**. The reasons are simple:

```text
train: Feb 2025 → Feb 2026   includes the smoggy Delhi winter (Oct-Jan)
test : Mar 2026 → Sep 2026   spring, summer and monsoon, when the air is cleaner
```

So the model learns on a **dirtier** period than the one it is tested on. This
is a real risk: a model trained mostly on winter may guess too high in summer.
We do not fix it in this ticket. We write it down now, so that when we look at
the plot in Step 5 we know what to look for.


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# X = the inputs the model looks at, y = the answer it must learn to predict
X_train = train[feature_names]
y_train = train["target"]
X_test = test[feature_names]
y_test = test["target"]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape, "| y_test :", y_test.shape)

# Two steps in a row: first scale, then Ridge. alpha stays at the default.
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge()),
])

# Learn from the TRAIN rows only
pipe.fit(X_train, y_train)
print("\nPipeline fitted.")

X_train: (283, 5) | y_train: (283,)
X_test : (145, 5) | y_test : (145,)

Pipeline fitted.


In [21]:
scaler = pipe.named_steps["scale"]

# Compare: what the scaler learned vs the mean of each set
mean_check = pd.DataFrame({
    "scaler learned": scaler.mean_,          # what the pipeline stored
    "train mean": X_train.mean().values,     # should be identical
    "test mean": X_test.mean().values,       # should be different
}, index=feature_names)

print(mean_check.round(2).to_string())


# The scaler's numbers must equal the TRAIN means, not the test means
assert (abs(scaler.mean_ - X_train.mean().values) < 1e-9).all(), "scaler did not learn from train"
print("\nCheck passed: the scaler's mean is the train mean.")


               scaler learned  train mean  test mean
pm25_until_17          108.91      108.91      55.53
lag_1                  114.28      114.28      57.47
mean_3                 113.36      113.36      57.94
mean_7                 111.89      111.89      59.13
month                    7.18        7.18       5.28

Check passed: the scaler's mean is the train mean.


In [22]:
# The pipeline scales X_test with the TRAIN mean and spread, then Ridge predicts
ridge_pred = pipe.predict(X_test)

ridge_check = pd.DataFrame({
    "today_until_17": X_test["pm25_until_17"],
    "real_tomorrow": y_test,
    "ridge_says": ridge_pred,
})

ridge_check["gap"] = ridge_check["ridge_says"] - ridge_check["real_tomorrow"]

print("Predictions made:", len(ridge_pred))
ridge_check.head(8).round(1)



Predictions made: 145


,today_until_17,real_tomorrow,ridge_says,gap
date,,,,
2026-03-01 00:00:00+05:30,83.6,100.0,92.0,-8.0
2026-03-02 00:00:00+05:30,93.8,95.5,95.8,0.3
2026-03-03 00:00:00+05:30,100.7,76.7,96.7,20.0
2026-03-04 00:00:00+05:30,78.9,68.8,84.7,15.9
2026-03-05 00:00:00+05:30,67.9,87.5,79.3,-8.2
2026-03-06 00:00:00+05:30,71.6,148.4,78.4,-70.0
2026-03-07 00:00:00+05:30,140.0,122.0,106.8,-15.3
2026-03-08 00:00:00+05:30,111.2,117.2,91.8,-25.5


## 4. Score all three methods on the same 145 days

### The one question this step answers

Asha's rule, the yesterday rule and Ridge all guess tomorrow's air for the same
145 test days. **Who is wrong by the least, and who warns us about Poor days?**

### Step 4.1: put everything on one row per day

Each row is one test day. One column is **the truth**, and the other three are
**guesses** from three different "people".

```text
date        actual        persistence   yesterday_full_day   ridge
            (THE TRUTH)   (guess)       (guess)              (guess)
            tomorrow's    today's       yesterday's          the trained
            real mean     00-17 mean    full-day mean        pipeline
Mar 1         99.9           83.6          111.3               97.2
Mar 2         95.5           93.8           87.3               ...
```

Read the Mar 1 row from left to right. It is 18:00 on Mar 1 and we are
forecasting **Mar 2**. The row is labelled with the day we forecast *from*, and
`actual` is what Mar 2 really turned out to be.

| Column | Truth or guess? | Where it comes from |
|---|---|---|
| `actual` | **truth** | `y_test`, which is `test["target"]`: tomorrow's real full-day mean |
| `persistence` | guess | `pm25_until_17`, today's 00-17 mean, the same number DAF-06 used |
| `yesterday_full_day` | guess | `lag_1`, yesterday's full-day mean, the same as DAF-06's `shift(1)` |
| `ridge` | guess | `ridge_pred`, from the pipeline |

**`actual` is the `target`.** It is the answer key. It is never given to any
method as an input: Ridge's five features all come from before 18:00 today, and
`target` is not among them. It sits in the table only so that we can mark the
guesses afterwards.

```text
score of a method = how far its guess is from the truth
                  = | actual - guess |,  averaged over the 145 days
```

The baselines are **re-scored on these 145 days**. They are not copied from
DAF-06's 161-day numbers, so all three are marked on the same exam paper.

### Step 4.2: the two scores

**MAE** (mean absolute error) is the average size of the mistake, in µg/m³.
Lower is better.

```text
actual 100, guess  90  ->  mistake 10
actual  80, guess 100  ->  mistake 20
MAE = (10 + 20) / 2 = 15
```

**Recall of Poor days** answers: *"Of the days that really were Poor-or-worse
(91 or more), how many did the method warn us about?"* A warning means the
guess was 91 or more. Every day falls into one of four boxes:

```text
                        guess >= 91 (warned)       guess < 91 (no warning)
actual >= 91 (Poor)     ✅ caught                   ❌ missed
actual <  91 (clean)    ⚠️ false alarm              ✔ correctly quiet

recall = caught / (caught + missed)
```

Example: 17 Poor days, of which 6 were caught and 11 missed. Then
recall = 6 / 17 = 0.35. The method warned about 35 out of every 100 bad days.

Why both scores? MAE treats a 10-point mistake on a clean day like a 10-point
mistake on a smog day. But a **missed Poor day is the mistake that hurts people**,
because someone went for a run in the smog. A model can improve MAE by being
good on easy days and still miss the dangerous ones. So the ticket asks for both.

### Step 4.3: the percentage change against persistence

The ticket asks whether Ridge beat persistence **by 10% on MAE**:

```text
change = (ridge MAE - persistence MAE) / persistence MAE x 100

persistence MAE = 20, ridge MAE = 17  ->  (17 - 20) / 20 = -15 %   (better)
persistence MAE = 20, ridge MAE = 21  ->  (21 - 20) / 20 = +5 %    (worse)
```

A negative number means Ridge is **better**. To pass, we need -10% or lower,
**and** recall must not fall.


In [23]:
from sklearn.metrics import mean_absolute_error

# One row per test day: the real answer, and each method's guess for it
compare = pd.DataFrame(index=test.index)
compare["actual"] = y_test                                # the real tomorrow
compare["persistence"] = X_test["pm25_until_17"]          # Asha's rule: tomorrow = today so far
compare["yesterday_full_day"] = X_test["lag_1"]           # tomorrow = yesterday's full day
compare["ridge"] = ridge_pred                             # the trained pipeline

method_names = ["persistence", "yesterday_full_day", "ridge"]

assert compare.notna().all().all(), "a guess is missing"
print("Days scored:", len(compare))
compare.head(5).round(1)


Days scored: 145


,actual,persistence,yesterday_full_day,ridge
date,,,,
2026-03-01 00:00:00+05:30,100.0,83.6,111.3,92.0
2026-03-02 00:00:00+05:30,95.5,93.8,87.3,95.8
2026-03-03 00:00:00+05:30,76.7,100.7,100.0,96.7
2026-03-04 00:00:00+05:30,68.8,78.9,95.5,84.7
2026-03-05 00:00:00+05:30,87.5,67.9,76.7,79.3


In [24]:
actual_poor = compare["actual"] >= poor_threshold        # True on days that really were Poor-or-worse
print("Poor-or-worse days in the test set:", actual_poor.sum())

results = []
for name in method_names:
    warned = compare[name] >= poor_threshold             # True on days this method warned

    caught = (actual_poor & warned).sum()                # ✅ Poor, and warned
    missed = (actual_poor & ~warned).sum()               # ❌ Poor, no warning
    false_alarms = (~actual_poor & warned).sum()         # ⚠️ clean, but warned

    results.append({
        "model": name,
        "mae": mean_absolute_error(compare["actual"], compare[name]),   # (real, guess)
        "caught": caught,
        "missed": missed,
        "false_alarms": false_alarms,
        "recall_poor": caught / (caught + missed),
    })

results = pd.DataFrame(results)

# The four boxes must add up to the Poor days, for every method
assert (results["caught"] + results["missed"] == actual_poor.sum()).all(), "boxes do not add up"

print(results.to_string(index=False, float_format="%.3f"))


Poor-or-worse days in the test set: 17
             model    mae  caught  missed  false_alarms  recall_poor
       persistence 16.344       8       9            12        0.471
yesterday_full_day 17.092       8       9            10        0.471
             ridge 15.507       6      11             7        0.353


In [ ]:
# ── Step 4.3  How much better or worse is each method than persistence? ────
# `results` has one row per method:
#
#       model                mae     recall_poor
#       persistence         16.34       0.471
#       yesterday_full_day  17.09       0.471
#       ridge               15.51       0.353
#
# Persistence is the free rule we are trying to beat, so it is the reference row.

# Step 1: pull the reference numbers out of the table.
#   results["model"] == "persistence"   -> True on the persistence row only
#   results.loc[that row, "mae"]        -> a one-item column: just persistence's MAE
#   .iloc[0]                            -> take that one item out as a plain number
persistence_mae = results.loc[results["model"] == "persistence", "mae"].iloc[0]
persistence_recall = results.loc[results["model"] == "persistence", "recall_poor"].iloc[0]

# Step 2: percentage change in MAE against persistence, for EVERY row at once.
#   change = (method MAE - persistence MAE) / persistence MAE x 100
#
#   negative -> the method's mistakes are SMALLER than persistence's (better)
#   positive -> the method's mistakes are BIGGER  than persistence's (worse)
#   Example: persistence 20, method 17 -> (17 - 20) / 20 x 100 = -15 %  (better)
#   The persistence row is always 0 %, because it is compared with itself.
results["mae_change_pct"] = (results["mae"] - persistence_mae) / persistence_mae * 100

# Step 3: change in recall. This is a plain subtraction, NOT a percentage.
#   positive -> the method warns about MORE Poor days than persistence (better)
#   negative -> the method warns about FEWER Poor days than persistence (worse)
#   Example: persistence 0.47, method 0.35 -> 0.35 - 0.47 = -0.12
#   (the method warns about 12 fewer Poor days in every 100)
results["recall_change"] = results["recall_poor"] - persistence_recall

# Step 4: show only the four columns we need to judge the ticket's question:
#   "10 % lower MAE (mae_change_pct <= -10) AND recall not lower (recall_change >= 0)?"
print(results[["model", "mae", "mae_change_pct", "recall_poor", "recall_change"]]
      .to_string(index=False, float_format="%.3f"))


16.343981382612807
             model    mae  mae_change_pct  recall_poor  recall_change
       persistence 16.344           0.000        0.471          0.000
yesterday_full_day 17.092           4.579        0.471          0.000
             ridge 15.507          -5.121        0.353         -0.118


## 5. Look at the guesses: where does each method go wrong?

### The one question this step answers

A table of averages says *how much* each method is wrong. It does not say
*when*. A picture shows the days where the guess and the truth pull apart, and
we can check whether Ridge and persistence fail on the **same** days.

### What the chart shows

```text
PM2.5 (µg/m³)
   │            ●  actual  (the truth: tomorrow's real mean)
150│         ╱╲
   │        ╱  ╲       ── ridge        (the trained pipeline's guess)
 91├ ─ ─ ─ ╱─ ─ ╲─ ─ ─  ── persistence  (today's 00-17 mean)
   │      ╱      ╲                      the dashed line = Poor-or-worse
 50│  ╱╲ ╱        ╲╱╲                   (91 and above, same line as DAF-06)
   └───────────────────────► test days, Mar 2026 → Sep 2026
```

Each dot on a line is **one forecast day**. The x-axis is the day we forecast
from, and the y-axis is the PM2.5 for the following day.

| Mark on the chart | Meaning |
|---|---|
| Line `actual` | what really happened |
| Line `ridge`, line `persistence` | what each method guessed |
| Dashed horizontal line at 91 | the Poor-or-worse line: above it, people are warned |
| Red ✖ | a **missed Poor day**: `actual` is above 91, but Ridge guessed below 91 |

The ticket says "the 90 line". DAF-06 defined Poor-or-worse as 91 or more, so
we keep 91 to stay comparable.

### How to read it

- Where the `ridge` line hugs `actual`, Ridge is right.
- A red ✖ is the mistake that matters most: real smog and no warning.
- If `ridge` and `persistence` jump at the same moments, both are making the
  same kind of mistake. That would mean a new model has to bring **new
  information**, not just a new formula.

### Step 5.2: the worst days, as a table

The chart is good at showing patterns and poor at giving exact numbers. So we
also list the eight days where Ridge was furthest from the truth, with
persistence's mistake on the same day:

```text
error = | guess - actual |          (always positive, in µg/m³)
```

If persistence also has a large error on those days, the day was simply
**hard for everyone**. If persistence was fine and only Ridge failed, the model
did something wrong there.


In [28]:
import plotly.graph_objects as go

# Days that really were Poor-or-worse, and days where Ridge did NOT warn
missed_by_ridge = compare[(compare["actual"] >= poor_threshold) & (compare["ridge"] < poor_threshold)]

fig = go.Figure()

# The truth, then each guess. Same colours for the same method on every chart.
for name in ["actual", "persistence", "ridge"]:
    fig.add_trace(go.Scatter(
        x=compare.index,
        y=compare[name],
        mode="lines+markers",
        name=name,
        line=dict(width=3 if name == "actual" else 1.5),
        marker=dict(size=4),
        hovertemplate=f"{name}: " + "%{y:.1f} µg/m³<br>Date: %{x|%Y-%m-%d}<extra></extra>",
    ))

# Red ✖ on every Poor day that Ridge failed to warn about (drawn at the truth)
fig.add_trace(go.Scatter(
    x=missed_by_ridge.index,
    y=missed_by_ridge["actual"],
    mode="markers",
    name=f"Poor day missed by Ridge ({len(missed_by_ridge)})",
    marker=dict(symbol="x", size=11, color="red", line=dict(width=2)),
))

# The Poor-or-worse line
fig.add_hline(y=poor_threshold, line_dash="dash", annotation_text=f"Poor-or-worse ({poor_threshold})")

fig.update_layout(
    title="Actual vs guesses on the test days (Mar 2026 to Sep 2026)",
    xaxis_title="Day the forecast is made (18:00)",
    yaxis_title="Next-day PM2.5 (µg/m³)",
    hovermode="x unified",
    height=500,
)
fig.show()

print("Poor days missed by Ridge:", len(missed_by_ridge))


Poor days missed by Ridge: 11


In [31]:
# error = how far each guess is from the truth, on each day (always positive)
errors = pd.DataFrame({
    "actual": compare["actual"],
    "ridge": compare["ridge"],
    "persistence": compare["persistence"],
    "ridge_error": (compare["ridge"] - compare["actual"]).abs(),
    "persistence_error": (compare["persistence"] - compare["actual"]).abs(),
})

# The 8 days where Ridge was wrong by the most
worst_days = errors.sort_values("ridge_error", ascending=False).head(8)
print(worst_days.round(1).to_string())

# Do the two methods fail on the same days? Compare their 10 worst days.
ridge_worst10 = set(errors.nlargest(10, "ridge_error").index)
persistence_worst10 = set(errors.nlargest(10, "persistence_error").index)
print("\nDays in BOTH methods' worst 10:", len(ridge_worst10 & persistence_worst10), "of 10")

# Does Ridge lean high or low? average (guess - actual): positive = guesses too high
print("\nAverage (guess - actual) over all test days:")
print((compare[["persistence", "ridge"]].sub(compare["actual"], axis=0)).mean().round(1).to_string())

# The same, only on the days that really were Poor
poor_days = compare[compare["actual"] >= poor_threshold]
print("\nAverage (guess - actual) on Poor days only:")
print((poor_days[["persistence", "ridge"]].sub(poor_days["actual"], axis=0)).mean().round(1).to_string())


                           actual  ridge  persistence  ridge_error  persistence_error
date                                                                                 
2026-03-06 00:00:00+05:30   148.4   78.4         71.6         70.0               76.8
2026-05-28 00:00:00+05:30    24.1   76.5         83.3         52.4               59.2
2026-04-18 00:00:00+05:30   120.5   74.2         73.5         46.3               47.0
2026-03-15 00:00:00+05:30    43.1   84.8         70.4         41.8               27.4
2026-07-05 00:00:00+05:30    32.7   68.8         72.7         36.0               39.9
2026-04-30 00:00:00+05:30    40.0   74.8         63.0         34.7               23.0
2026-04-01 00:00:00+05:30    89.2   54.7         32.7         34.5               56.5
2026-07-20 00:00:00+05:30    18.3   52.4         41.6         34.1               23.4

Days in BOTH methods' worst 10: 5 of 10

Average (guess - actual) over all test days:
persistence   -1.4
ridge          4.4

Average (guess